## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-09-20-52-03 +0000,nyt,The Nvidia Affair,https://www.nytimes.com/2026/11/09/world/nvidi...
1,2026-11-09-20-51-50 +0000,nypost,GLP-1 drugs must have new warning label over i...,https://nypost.com/2026/11/09/health/fda-issue...
2,2026-11-09-20-50-18 +0000,wapo,FDA tells GLP-1 makers to warn of potential vi...,https://www.washingtonpost.com/health/2026/11/...
3,2026-11-09-20-50-12 +0000,nyt,Shipwreck Off Senegal Kills Dozens as Canary I...,https://www.nytimes.com/2026/11/09/world/afric...
4,2026-11-09-20-50-00 +0000,wsj,Oil Futures Slide as Trump Says New Strikes on...,https://www.wsj.com/finance/commodities-future...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2662/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
33,trump,44
0,nvidia,38
111,huang,28
110,jensen,23
109,ceo,21


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
14,2026-11-09-20-39-13 +0000,nypost,Trader warned of Nvidia CEO Jensen Huang’s ‘un...,https://nypost.com/2026/11/09/business/trader-...,194
50,2026-11-09-19-47-46 +0000,nypost,Nvidia CEO Jensen Huang has no obvious success...,https://nypost.com/2026/11/09/business/nvidia-...,185
180,2026-11-09-14-57-09 +0000,nypost,Nvidia stock plunges 5% after CEO Jensen Huang...,https://nypost.com/2026/11/09/business/nvidia-...,181
48,2026-11-09-19-53-19 +0000,nypost,"Bill Ackman, Elon Musk back Nvidia CEO Jensen ...",https://nypost.com/2026/11/09/business/bill-ac...,180
331,2026-11-09-00-30-00 +0000,wsj,President Trump said Sunday that he was joking...,https://www.wsj.com/politics/policy/trump-says...,171


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
14,194,2026-11-09-20-39-13 +0000,nypost,Trader warned of Nvidia CEO Jensen Huang’s ‘un...,https://nypost.com/2026/11/09/business/trader-...
331,171,2026-11-09-00-30-00 +0000,wsj,President Trump said Sunday that he was joking...,https://www.wsj.com/politics/policy/trump-says...
316,79,2026-11-09-02-00-00 +0000,wsj,Voters fed up with high prices gave Democrats ...,https://www.wsj.com/politics/elections/democra...
54,58,2026-11-09-19-38-13 +0000,nypost,Rep. Thomas Massie reintroduces impeachment ar...,https://nypost.com/2026/11/09/us-news/rep-thom...
232,46,2026-11-09-11-48-00 +0000,wsj,This week will pose two major tests for Presid...,https://www.wsj.com/politics/policy/nvidias-je...
109,43,2026-11-09-18-12-15 +0000,nyt,"Suspect in Mass Shooting in Krakow, Poland, Di...",https://www.nytimes.com/2026/11/08/world/europ...
17,43,2026-11-09-20-37-19 +0000,latimes,Supreme Court OKs border wall construction in ...,https://www.latimes.com/politics/story/2026-11...
191,42,2026-11-09-14-28-31 +0000,nypost,Russell Brand sentenced to over 10 years in pr...,https://nypost.com/2026/11/09/world-news/russe...
152,40,2026-11-09-16-38-04 +0000,chitri,Nvidia’s investigation into CEO Jensen Huang’s...,https://www.chicagotribune.com/2026/11/09/nvid...
188,40,2026-11-09-14-33-05 +0000,wapo,Stephen Miller starts job as chief of staff; G...,https://www.washingtonpost.com/politics/2026/1...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
